# 🚀 FabrIQ Fabric Defect Detection - YOLO Training (Google Colab)

This notebook trains YOLOv8 models for fabric defect detection using Google Colab's free GPU.

## 📋 Steps:
1. Setup environment and install dependencies
2. Upload and organize dataset
3. Create annotations (for detection)
4. Train YOLO model
5. Evaluate and download results

**Note**: This notebook is designed to work with VS Code's Colab extension. Make sure you have the extension installed and connected to Google Colab.


## Kaggle Quick Start (YOLO detection, dataset already prepared)

Use this section if you already have a YOLO-formatted dataset at `/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset` (train/images, train/labels, val/images, val/labels).

Steps:
1. Run the install cell (next) if ultralytics is missing
2. Run the Kaggle detection cell below
3. Results will be saved to `/kaggle/working/runs/detect/fabriq_detection/`



In [ ]:
# Kaggle detection training with existing YOLO dataset (copies to /kaggle/working and cleans corrupt JPGs)
from pathlib import Path
import yaml
import shutil
import cv2
import numpy as np
from ultralytics import YOLO

INPUT_ROOT = Path('/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset')
WORK_ROOT = Path('/kaggle/working/FabrIQ_YOLO_Detection_Dataset')

# Verify input structure
for p in [INPUT_ROOT / 'train' / 'images', INPUT_ROOT / 'train' / 'labels', INPUT_ROOT / 'val' / 'images', INPUT_ROOT / 'val' / 'labels']:
    assert p.exists(), f"Missing path: {p}"

# Copy once to working dir to allow cleaning
if not WORK_ROOT.exists():
    shutil.copytree(INPUT_ROOT, WORK_ROOT)
    print(f"✅ Copied dataset to {WORK_ROOT}")
else:
    print(f"ℹ️ Working copy already exists: {WORK_ROOT}")

# Define classes
CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

# Clean/correct corrupt JPGs (attempt re-encode, else delete with matching label)
def is_image_ok(path: Path) -> bool:
    try:
        data = np.fromfile(path, dtype=np.uint8)
        if data.size == 0:
            return False
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img is None:
            return False
        # Re-encode to ensure readable
        cv2.imencode(path.suffix, img)[1].tofile(path)
        return True
    except Exception:
        return False

bad = 0
for split in ['train', 'val']:
    img_dir = WORK_ROOT / split / 'images'
    lbl_dir = WORK_ROOT / split / 'labels'
    for img_path in img_dir.glob('*.jpg'):
        ok = is_image_ok(img_path)
        if not ok:
            bad += 1
            label_path = lbl_dir / (img_path.stem + '.txt')
            img_path.unlink(missing_ok=True)
            label_path.unlink(missing_ok=True)
print(f"🧹 Cleaned corrupt images: removed {bad} files (if any)")

# Create data.yaml in working dir
yaml_path = Path('/kaggle/working/fabriq_detection_data.yaml')
data_config = {
    'path': str(WORK_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CLASSES),
    'names': {i: cls for i, cls in enumerate(CLASSES)}
}
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ data.yaml written to: {yaml_path}")
print(f"Dataset path: {WORK_ROOT}")

# Train
print("\n🚀 Starting Detection Training (Kaggle ready dataset)...")
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    project='/kaggle/working/runs/detect',
    name='fabriq_detection',
    exist_ok=True,
    patience=20,
    save=True,
    plots=True
)

print("\n✅ Training Complete!")
print(f"📊 Results: /kaggle/working/runs/detect/fabriq_detection/")


## Step 1: Install Dependencies


In [ ]:
# Install required packages
%pip install ultralytics torch torchvision opencv-python pillow numpy pyyaml tqdm -q

# Verify installation
import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


## Step 2: Upload Dataset

**Option A: Upload from your computer**


In [ ]:
# Upload dataset zip file
from google.colab import files
import zipfile
import os

print("📤 Upload your dataset zip file...")
uploaded = files.upload()

# Extract if zip file
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"📦 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("✅ Extraction complete!")


**Option B: Use Google Drive**


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset from Drive
import shutil
drive_path = '/content/drive/MyDrive/FabrIQ_dataset'  # Change this to your path
if os.path.exists(drive_path):
    shutil.copytree(drive_path, '/content/FabrIQ_dataset', dirs_exist_ok=True)
    print("✅ Dataset copied from Drive")
else:
    print("⚠️ Path not found. Update drive_path variable.")


## Step 3: Organize Dataset


In [ ]:
import os
import shutil
from pathlib import Path
import random
from collections import defaultdict

# Configuration
OUTPUT_FOLDER_NAME = 'FabrIQ_Final_Dataset'
TRAIN_RATIO = 0.8
SEED = 42

# 20 Target Classes
TARGET_CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

# Class mapping
CLASS_MAPPING = {
    'normal': 'normal', 'good': 'normal', 'defect free': 'normal', 'no defect': 'normal',
    'hole': 'knit hole', 'holes': 'knit hole', 'knit hote': 'knit hole',
    'oil spot': 'oil spot', 'stain': 'oil spot',
    'lines': 'oil lines', 'horizontal': 'oil lines', 'vertical': 'oil lines',
    'thread error': 'pulling thread', 'thread': 'pulling thread',
    'yarn missing': 'yarn variation', 'mix yam': 'mix yarn',
}

def normalize_name(name):
    return name.lower().strip().replace('_', ' ').replace('-', ' ')

def find_target_class(folder_name, file_name, full_path):
    path_str = str(full_path).lower()
    
    if any(x in path_str for x in ['nodefect', 'no defect', 'defect free', 'good']):
        return 'normal'
    
    f_name_norm = normalize_name(folder_name)
    sorted_keys = sorted(CLASS_MAPPING.keys(), key=len, reverse=True)
    
    for key in sorted_keys:
        if key in f_name_norm:
            return CLASS_MAPPING[key]
    
    if 'defect' in path_str:
        return 'mix yarn'
    
    return None

# Organize dataset
print("🚀 Starting dataset organization...")
random.seed(SEED)

root_dir = Path('/content')
output_dir = root_dir / OUTPUT_FOLDER_NAME

# Create structure
for split in ['train', 'val']:
    (output_dir / split / 'images').mkdir(parents=True, exist_ok=True)

# Find images
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
found_files = defaultdict(list)

for path in root_dir.rglob('*'):
    if OUTPUT_FOLDER_NAME in str(path):
        continue
    if path.is_file() and path.suffix.lower() in valid_exts:
        target = find_target_class(path.parent.name, path.name, path)
        if target:
            found_files[target].append(path)

# Process files
print("\n📊 Data Distribution:")
total_processed = 0

for cls in TARGET_CLASSES:
    images = found_files[cls]
    random.shuffle(images)
    
    split_idx = int(len(images) * TRAIN_RATIO)
    splits = {'train': images[:split_idx], 'val': images[split_idx:]}
    
    print(f"   {cls}: {len(images)} images")
    
    for split_name, split_imgs in splits.items():
        for idx, img_path in enumerate(split_imgs):
            clean_cls_name = cls.replace(' ', '_')
            new_name = f"FabrIQ_{clean_cls_name}_{idx+1:05d}{img_path.suffix}"
            dest = output_dir / split_name / 'images' / new_name
            shutil.copy2(img_path, dest)
            total_processed += 1

print(f"\n✅ Processed {total_processed} images")
print(f"📂 Dataset at: {output_dir}")


In [ ]:
# Prepare classification dataset
from pathlib import Path
import shutil

yolo_dataset = Path('/content/FabrIQ_YOLO_Dataset')
source_dataset = Path('/content/FabrIQ_Final_Dataset')

CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

for split in ['train', 'val']:
    source_dir = source_dataset / split / 'images'
    
    # Create class folders
    for class_name in CLASSES:
        class_folder = yolo_dataset / split / class_name.replace(' ', '_')
        class_folder.mkdir(parents=True, exist_ok=True)
    
    # Copy images to class folders
    count = 0
    for img_file in source_dir.glob("*.jpg"):
        filename = img_file.stem
        parts = filename.split('_')
        
        if len(parts) >= 2 and parts[0] == 'FabrIQ':
            class_parts = parts[1:-1]
            class_name = '_'.join(class_parts)
            class_name_with_spaces = class_name.replace('_', ' ')
            
            if class_name_with_spaces in CLASSES:
                dest_folder = yolo_dataset / split / class_name
                shutil.copy2(img_file, dest_folder / img_file.name)
                count += 1
    
    print(f"✅ {split}: {count} images organized")

print(f"\n📁 Classification dataset ready at: {yolo_dataset}")


In [ ]:
# Train Classification Model
from ultralytics import YOLO

print("🚀 Starting Classification Training...")

model = YOLO('yolov8n-cls.pt')

results = model.train(
    data=str(yolo_dataset),
    epochs=50,
    imgsz=224,
    batch=16,
    device=0,  # Use GPU
    project='/content/runs/classify',
    name='fabriq_classification',
    exist_ok=True,
    patience=20,
    save=True,
    plots=True
)

print("\n✅ Training Complete!")
print(f"📊 Results: {Path('/content/runs/classify/fabriq_classification')}")


### Option B: Detection (Bounding Boxes)


In [ ]:
# Create pseudo-annotations for detection
from pathlib import Path
import shutil

DATASET_PATH = Path('/content/FabrIQ_Final_Dataset')
OUTPUT_PATH = Path('/content/FabrIQ_YOLO_Detection_Dataset')

CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

print("📝 Creating pseudo-annotations...")

# Create structure
for split in ['train', 'val']:
    (OUTPUT_PATH / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_PATH / split / 'labels').mkdir(parents=True, exist_ok=True)

total_annotations = 0

for split in ['train', 'val']:
    source_dir = DATASET_PATH / split / 'images'
    count = 0
    
    for img_file in source_dir.glob("*.jpg"):
        filename = img_file.stem
        parts = filename.split('_')
        
        if len(parts) >= 2 and parts[0] == 'FabrIQ':
            class_parts = parts[1:-1]
            class_name = '_'.join(class_parts)
            class_name_with_spaces = class_name.replace('_', ' ')
            
            if class_name_with_spaces in CLASSES:
                # Copy image
                dest_img = OUTPUT_PATH / split / 'images' / img_file.name
                shutil.copy2(img_file, dest_img)
                
                # Create annotation (full image box)
                class_id = CLASSES.index(class_name_with_spaces)
                annotation = f"{class_id} 0.5 0.5 1.0 1.0\n"
                
                label_file = OUTPUT_PATH / split / 'labels' / f"{img_file.stem}.txt"
                with open(label_file, 'w') as f:
                    f.write(annotation)
                
                count += 1
                total_annotations += 1
    
    print(f"✅ {split}: {count} annotations created")

print(f"\n✅ Total: {total_annotations} annotations")
print(f"📁 Dataset ready at: {OUTPUT_PATH}")


In [ ]:
# Create data.yaml for detection
import yaml

CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

data_config = {
    'path': '/content/FabrIQ_YOLO_Detection_Dataset',
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CLASSES),
    'names': {i: cls for i, cls in enumerate(CLASSES)}
}

yaml_path = Path('/content/fabriq_detection_data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Created data.yaml: {yaml_path}")


In [ ]:
# Train Detection Model
from ultralytics import YOLO

print("🚀 Starting Detection Training...")

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/fabriq_detection_data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,  # Use GPU
    project='/content/runs/detect',
    name='fabriq_detection',
    exist_ok=True,
    patience=20,
    save=True,
    plots=True
)

print("\n✅ Training Complete!")
print(f"📊 Results: {Path('/content/runs/detect/fabriq_detection')}")


## Step 5: Download Results


In [ ]:
# Download trained model and results
from google.colab import files
from pathlib import Path
import zipfile

# Create zip file with results
results_dir = Path('/content/runs')
zip_path = Path('/content/fabriq_results.zip')

if results_dir.exists():
    print("📦 Creating zip file...")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in results_dir.rglob('*'):
            if file_path.is_file():
                zipf.write(file_path, file_path.relative_to(results_dir.parent))
    
    print(f"✅ Created: {zip_path}")
    print("\n📥 Downloading...")
    files.download(str(zip_path))
    
    # Also download best model separately
    best_model = Path('/content/runs/classify/fabriq_classification/weights/best.pt')
    if best_model.exists():
        files.download(str(best_model))
        print("✅ Best model downloaded")
else:
    print("⚠️ Results directory not found")


## 📊 Training Summary

After training completes:

1. **Check Results**: `/content/runs/classify/fabriq_classification/` or `/content/runs/detect/fabriq_detection/`
2. **Best Model**: `weights/best.pt`
3. **Training Curves**: `results.png`
4. **Confusion Matrix**: `confusion_matrix.png`

## 💡 Tips

- **GPU Time**: Training takes 30-60 minutes on Colab GPU
- **Save Progress**: Colab sessions disconnect after inactivity
- **Download Models**: Always download your trained models before disconnecting
- **Monitor**: Check training progress in the output cells

## 🎉 Done!

Your model is trained and ready to use!
